In [0]:
%run /Workspace/Users/vnvarkhede@gmail.com/databricks-genai-data-analyst-copilot/notebooks/09_pipeline/01_pipeline_config.py

In [0]:
# ================================================================
# PHASE 20 — PRODUCTION DATA PIPELINE
# FILE: 02_data_pipeline.py
# ================================================================

print("=" * 70)
print("PHASE 20 — PRODUCTION DATA PIPELINE")
print("=" * 70)


In [0]:
# ================================================================
# CELL 3 — IMPORTS
# ================================================================

import json
import time

from pyspark.sql import functions as F

print("Imports successful.")

In [0]:
# ================================================================
# CELL 4 — PIPELINE EXECUTION CONFIGURATION
# ================================================================

DATA_PIPELINE_CONFIG = {
    "bronze_table": BRONZE_TABLE,
    "silver_table": SILVER_TABLE,
    "gold_table": GOLD_TABLE,
    "catalog": CATALOG
}

print(
    json.dumps(
        DATA_PIPELINE_CONFIG,
        indent=2
    )
)

In [0]:
# ================================================================
# CELL 5 — BRONZE VALIDATION
# ================================================================

print("=" * 70)
print("BRONZE LAYER VALIDATION")
print("=" * 70)

bronze_start = time.time()

# Use the actual bronze table name (sales_raw, not sales)
bronze_table_name = f"{CATALOG}.bronze.sales_raw"

bronze_df = spark.table(
    bronze_table_name
)

bronze_count = bronze_df.count()

bronze_columns = bronze_df.columns

bronze_execution_ms = round(
    (time.time() - bronze_start) * 1000,
    2
)

print(
    "Bronze table:",
    bronze_table_name
)

print(
    "Bronze rows:",
    bronze_count
)

print(
    "Bronze columns:",
    bronze_columns
)

print(
    "Execution time:",
    bronze_execution_ms,
    "ms"
)

if bronze_count == 0:

    raise RuntimeError(
        "Bronze table contains zero rows."
    )

print()
print("Bronze validation: PASS")

In [0]:
# ================================================================
# CELL 6 — SILVER VALIDATION
# ================================================================

print("=" * 70)
print("SILVER LAYER VALIDATION")
print("=" * 70)

silver_start = time.time()

silver_df = spark.table(
    SILVER_TABLE
)

silver_count = silver_df.count()

silver_columns = silver_df.columns

silver_execution_ms = round(
    (time.time() - silver_start) * 1000,
    2
)

print(
    "Silver table:",
    SILVER_TABLE
)

print(
    "Silver rows:",
    silver_count
)

print(
    "Silver columns:",
    silver_columns
)

print(
    "Execution time:",
    silver_execution_ms,
    "ms"
)

if silver_count == 0:

    raise RuntimeError(
        "Silver table contains zero rows."
    )

print()
print("Silver validation: PASS")

In [0]:
# ================================================================
# CELL 7 — GOLD VALIDATION
# ================================================================

print("=" * 70)
print("GOLD LAYER VALIDATION")
print("=" * 70)

gold_start = time.time()

gold_df = spark.table(
    GOLD_TABLE
)

gold_count = gold_df.count()

gold_columns = gold_df.columns

gold_execution_ms = round(
    (time.time() - gold_start) * 1000,
    2
)

print(
    "Gold table:",
    GOLD_TABLE
)

print(
    "Gold rows:",
    gold_count
)

print(
    "Gold columns:",
    gold_columns
)

print(
    "Execution time:",
    gold_execution_ms,
    "ms"
)

if gold_count == 0:

    raise RuntimeError(
        "Gold table contains zero rows."
    )

print()
print("Gold validation: PASS")

In [0]:
# ================================================================
# CELL 8 — GOLD BUSINESS SCHEMA VALIDATION
# ================================================================

print("=" * 70)
print("GOLD BUSINESS SCHEMA VALIDATION")
print("=" * 70)

required_gold_columns = [
    "region",
    "total_revenue"
]

missing_gold_columns = [
    column
    for column in required_gold_columns
    if column not in gold_columns
]

for column in required_gold_columns:

    print(
        f"{'PASS' if column in gold_columns else 'FAIL'} - "
        f"{column}"
    )

if missing_gold_columns:

    raise RuntimeError(
        "Gold table is missing required columns: "
        + ", ".join(missing_gold_columns)
    )

print()
print("Gold business schema: PASS")

In [0]:
# ================================================================
# CELL 9 — GOLD DATA VALIDATION
# ================================================================

print("=" * 70)
print("GOLD DATA VALIDATION")
print("=" * 70)

null_region_count = (
    gold_df
    .filter(
        F.col("region").isNull()
    )
    .count()
)

null_revenue_count = (
    gold_df
    .filter(
        F.col("total_revenue").isNull()
    )
    .count()
)

negative_revenue_count = (
    gold_df
    .filter(
        F.col("total_revenue") < 0
    )
    .count()
)

print(
    "Null regions:",
    null_region_count
)

print(
    "Null revenue:",
    null_revenue_count
)

print(
    "Negative revenue:",
    negative_revenue_count
)

if null_region_count > 0:

    raise RuntimeError(
        "Gold table contains null region values."
    )

if null_revenue_count > 0:

    raise RuntimeError(
        "Gold table contains null total_revenue values."
    )

print()
print("Gold data validation: PASS")

In [0]:
# ================================================================
# CELL 10 — BUSINESS OUTPUT VALIDATION
# ================================================================

print("=" * 70)
print("BUSINESS OUTPUT VALIDATION")
print("=" * 70)

top_regions = (
    gold_df
    .select(
        "region",
        "total_revenue"
    )
    .orderBy(
        F.col("total_revenue").desc()
    )
    .limit(10)
)

display(top_regions)

top_region = (
    top_regions
    .first()
)

if top_region is None:

    raise RuntimeError(
        "Unable to determine highest revenue region."
    )

print()
print(
    "Highest revenue region:",
    top_region["region"]
)

print(
    "Highest revenue:",
    top_region["total_revenue"]
)

print()
print("Business output validation: PASS")

In [0]:
# ================================================================
# CELL 11 — PIPELINE METRICS
# ================================================================

PIPELINE_METRICS = {

    "bronze": {
        "table": BRONZE_TABLE,
        "row_count": bronze_count
    },

    "silver": {
        "table": SILVER_TABLE,
        "row_count": silver_count
    },

    "gold": {
        "table": GOLD_TABLE,
        "row_count": gold_count
    },

    "highest_revenue_region": top_region["region"],

    "highest_revenue": float(
        top_region["total_revenue"]
    )
}

print(
    json.dumps(
        PIPELINE_METRICS,
        indent=2,
        default=str
    )
)

In [0]:
# ================================================================
# CELL 12 — FINAL DATA PIPELINE VALIDATION
# ================================================================

print("=" * 70)
print("PHASE 20 — DATA PIPELINE VALIDATION")
print("=" * 70)

pipeline_checks = [

    (
        "Bronze available",
        bronze_count > 0
    ),

    (
        "Silver available",
        silver_count > 0
    ),

    (
        "Gold available",
        gold_count > 0
    ),

    (
        "Gold region column",
        "region" in gold_columns
    ),

    (
        "Gold revenue column",
        "total_revenue" in gold_columns
    ),

    (
        "No null regions",
        null_region_count == 0
    ),

    (
        "No null revenue",
        null_revenue_count == 0
    ),

    (
        "Highest revenue region available",
        top_region is not None
    )
]

failed_checks = []

for check_name, passed in pipeline_checks:

    print(
        f"{'PASS' if passed else 'FAIL'} - "
        f"{check_name}"
    )

    if not passed:

        failed_checks.append(
            check_name
        )

print()
print(
    "Total checks:",
    len(pipeline_checks)
)

print(
    "Failed checks:",
    len(failed_checks)
)

if failed_checks:

    raise RuntimeError(
        "Data pipeline validation failed: "
        + ", ".join(failed_checks)
    )

print()
print(
    "PHASE 20 DATA PIPELINE: PASS ✓"
)

In [0]:
# ================================================================
# CELL 13 — PIPELINE SUMMARY
# ================================================================

print("=" * 70)
print("PRODUCTION DATA PIPELINE SUMMARY")
print("=" * 70)

print()
print("BRONZE")
print("Table:", BRONZE_TABLE)
print("Rows:", bronze_count)

print()
print("SILVER")
print("Table:", SILVER_TABLE)
print("Rows:", silver_count)

print()
print("GOLD")
print("Table:", GOLD_TABLE)
print("Rows:", gold_count)

print()
print("BUSINESS RESULT")
print(
    "Highest revenue region:",
    top_region["region"]
)

print(
    "Highest revenue:",
    top_region["total_revenue"]
)

print()
print("=" * 70)
print("DATA PIPELINE STATUS: PASS ✓")
print("=" * 70)